# EmailGuard: AI-Powered Email Spam Detection System
### End-to-End NLP & Machine Learning Experimentation Notebook

This notebook documents the complete Machine Learning and Natural Language Processing lifecycle:
1. **Dataset Loading & Exploratory Data Analysis (EDA)**
2. **Text Preprocessing & Normalization**
3. **TF-IDF Feature Extraction (Unigram + Bigram)**
4. **Comparative Model Training** (MultinomialNB, Logistic Regression, LinearSVC)
5. **Cross-Validation & Held-Out Test Evaluation**
6. **Confusion Matrices & ROC-AUC Analysis**
7. **Model Serialization & Explainability**

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Ensure project backend modules can be imported
sys.path.append('../backend')
from ml.preprocessing import preprocess_text, extract_email_features
print("Environment initialized successfully.")

## 1. Load Dataset & Exploratory Data Analysis

In [2]:
dataset_path = Path('../dataset/spam_dataset.csv')
df = pd.read_csv(dataset_path)
print(f"Total clean samples: {len(df)}")
print("Class distribution:")
print(df['label'].value_counts())
print("\nSample records:")
df.head()

## 2. Text Preprocessing & NLP Pipeline

In [3]:
sample_email = "CONGRATULATIONS!! You won a $1,000 Walmart Gift Card! Visit http://scam.cc to claim now!"
cleaned = preprocess_text(sample_email)
features = extract_email_features(sample_email)

print("Raw Email:", sample_email)
print("Cleaned Text:", cleaned)
print("Extracted Explainability Features:", features)

## 3. TF-IDF Feature Extraction & Train-Test Split

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

df['cleaned_text'] = df['text'].apply(preprocess_text)
df['target'] = df['label'].map(lambda x: 1 if 'spam' in str(x).lower() else 0)

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['target'], test_size=0.20, random_state=42, stratify=df['target']
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Train matrix shape: {X_train_tfidf.shape}")
print(f"Test matrix shape: {X_test_tfidf.shape}")

## 4. Train & Compare Multiple Machine Learning Models

In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42),
    'Linear Support Vector Machine': CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=3000, class_weight='balanced', random_state=42), cv=3)
}

results = {}
for name, clf in models.items():
    clf.fit(X_train_tfidf, y_train)
    y_pred = clf.predict(X_test_tfidf)
    y_proba = clf.predict_proba(X_test_tfidf)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results[name] = {
        'Accuracy': round(acc * 100, 2),
        'Precision': round(prec * 100, 2),
        'Recall': round(rec * 100, 2),
        'F1-Score': round(f1 * 100, 2),
        'ROC-AUC': round(auc, 4)
    }

pd.DataFrame(results).T

## 5. Inference & Explainability Test

In [6]:
from ml.predict import predict_single

test_email = "URGENT: Your account has been suspended due to security violations. Click http://bank-verify.cc"
prediction = predict_single("Security Notice", test_email)
print("Prediction:", prediction['classification'])
print("Confidence:", prediction['confidence'], "%")
print("Risk Level:", prediction['risk_level'])
print("Explanation:", prediction['explanation'])